# Customer Segmentation EDA & Modeling Report
### B.Tech CSE Final Year Project

This notebook covers the Exploratory Data Analysis (EDA) and the K-Means Clustering modeling process used for segmenting retail customers based on their RFM metrics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

df = pd.read_csv('../data/online_retail.csv', parse_dates=['InvoiceDate'])
df.head()

## 1. Feature Engineering: RFM Metrics
Recency, Frequency, and Monetary metrics are calculated from the raw transactional data.

In [ ]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'count',
    'UnitPrice': lambda x: (x * df.loc[x.index, 'Quantity']).sum()
})
rfm.columns = ['Recency', 'Frequency', 'Monetary']
rfm.describe()

## 2. Determining Optimal K (Elbow Method)
We use the within-cluster sum of squares (WCSS) to find the 'elbow'.

In [ ]:
scaler = StandardScaler()
scaled_data = scaler.fit_transform(rfm)

wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42)
    kmeans.fit(scaled_data)
    wcss.append(kmeans.inertia_)

plt.plot(range(1, 11), wcss)
plt.title('Elbow Method')
plt.xlabel('Number of clusters')
plt.ylabel('WCSS')
plt.show()